# Chapter 89 - Loading and Generating Later

Saving is useful only when a later process can reconstruct the same tokenizer, architecture, and tensor state.

This chapter starts from files, builds one validated loader, and generates through a reusable inference interface.

## Learning goals

By the end of this chapter, you will be able to:

- locate and validate the files required for inference;

- reconstruct a tokenizer and model from versioned JSON configuration;

- load tensor state on CPU with restricted deserialization;

- verify vocabulary, state-key, and output-shape compatibility;

- encode, crop, generate, and decode without losing the displayed prompt;

- make sampling reproducible without mutating global random state;

- diagnose architecture mismatch and same-size tokenizer mismatch; and

- use one clean loader-and-generation interface in a later session.

## What “later” means in this notebook

Chapter 88 used a temporary directory and removed it so repository validation would not leave generated artifacts behind.

To keep this notebook standalone, a fixture function first creates the same four-file package inside another temporary directory.

The producer's model, tokenizer, optimizer, and configs are local variables that disappear when the function returns.

The section marked **Later session begins** uses only the artifact directory and reusable class code, which a real project would import from version-controlled modules.

## Required inference files

The Chapter 88 package uses:

- `model_state.pt` for parameters and persistent buffers;

- `tokenizer.json` for exact token meanings;

- `model_config.json` for the architecture; and

- optional `training_config.json` for documentation.

The training config is not required for generation and is not enough for exact training resume.

## Reuse the versioned tokenizer code

Loading reconstructs the saved mapping instead of training a new tokenizer.

The schema checks reject noncontiguous or duplicate IDs before they can reach the model.

In [1]:
class SaveableCharacterTokenizer:
    token_to_id: dict[str, int]
    id_to_token: dict[int, str]
    is_trained: bool

    def __init__(self) -> None:
        self.token_to_id = {}
        self.id_to_token = {}
        self.is_trained = False

    def train(self, text: str) -> None:
        if not text:
            raise ValueError("Training text must not be empty.")
        self.token_to_id = {
            token: token_id for token_id, token in enumerate(sorted(set(text)))
        }
        self.id_to_token = {
            token_id: token for token, token_id in self.token_to_id.items()
        }
        self.is_trained = True

    def _check_trained(self) -> None:
        if not self.is_trained:
            raise RuntimeError("Tokenizer must be trained before use.")

    @property
    def vocabulary_size(self) -> int:
        self._check_trained()
        return len(self.token_to_id)

    def encode(self, text: str) -> list[int]:
        self._check_trained()
        unknown_tokens = sorted(set(text) - set(self.token_to_id))
        if unknown_tokens:
            raise ValueError(f"Unknown tokens: {unknown_tokens!r}.")
        return [self.token_to_id[token] for token in text]

    def decode(self, token_ids: list[int]) -> str:
        self._check_trained()
        return "".join(self.id_to_token[token_id] for token_id in token_ids)

    def to_dict(self) -> dict[str, object]:
        self._check_trained()
        return {
            "format_version": 1,
            "tokenizer_type": "character",
            "token_to_id": self.token_to_id,
        }

    @classmethod
    def from_dict(
        cls,
        tokenizer_data: dict[str, object],
    ) -> "SaveableCharacterTokenizer":
        expected_keys = {"format_version", "tokenizer_type", "token_to_id"}
        if set(tokenizer_data) != expected_keys:
            raise ValueError("Tokenizer JSON has missing or unexpected fields.")
        if tokenizer_data["format_version"] != 1:
            raise ValueError("Unsupported tokenizer format version.")
        if tokenizer_data["tokenizer_type"] != "character":
            raise ValueError("Expected a character tokenizer.")
        raw_mapping = tokenizer_data["token_to_id"]
        if not isinstance(raw_mapping, dict) or not raw_mapping:
            raise TypeError("token_to_id must be a non-empty dictionary.")

        token_to_id: dict[str, int] = {}
        for token, token_id in raw_mapping.items():
            if not isinstance(token, str) or type(token_id) is not int:
                raise TypeError("Tokenizer entries must map strings to integer IDs.")
            token_to_id[token] = token_id
        expected_ids = list(range(len(token_to_id)))
        if sorted(token_to_id.values()) != expected_ids:
            raise ValueError("Tokenizer IDs must be unique and contiguous from zero.")

        tokenizer = cls()
        tokenizer.token_to_id = token_to_id
        tokenizer.id_to_token = {
            token_id: token for token, token_id in token_to_id.items()
        }
        tokenizer.is_trained = True
        return tokenizer

## Reuse the matching model code

A state dictionary stores named tensors, not the Python operations that connect them.

The class and attribute names below match the package producer exactly.

In [2]:
import math  # noqa: I001
import torch


class MultiHeadCausalSelfAttention(torch.nn.Module):
    embedding_dimension: int
    number_of_attention_heads: int
    head_size: int

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.head_size = embedding_dimension // number_of_attention_heads
        self.query_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.output_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension
        )
        self.attention_dropout = torch.nn.Dropout(dropout_rate)
        self.output_dropout = torch.nn.Dropout(dropout_rate)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = input_values.shape

        def split_heads(values: torch.Tensor) -> torch.Tensor:
            return values.view(
                batch_size,
                sequence_length,
                self.number_of_attention_heads,
                self.head_size,
            ).transpose(1, 2)

        queries = split_heads(self.query_projection(input_values))
        keys = split_heads(self.key_projection(input_values))
        values = split_heads(self.value_projection(input_values))
        attention_scores = queries @ keys.transpose(-2, -1)
        attention_scores = attention_scores / math.sqrt(self.head_size)
        causal_mask = self.causal_mask[:sequence_length, :sequence_length]
        attention_scores = attention_scores.masked_fill(~causal_mask, float("-inf"))
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = self.attention_dropout(attention_weights) @ values
        concatenated_values = (
            attended_values.transpose(1, 2)
            .contiguous()
            .view(batch_size, sequence_length, self.embedding_dimension)
        )
        projected_values = self.output_projection(concatenated_values)
        output_values: torch.Tensor = self.output_dropout(projected_values)
        return output_values

In [3]:
class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, embedding_dimension: int, dropout_rate: float) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, 4 * embedding_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(4 * embedding_dimension, embedding_dimension),
            torch.nn.Dropout(dropout_rate),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        output_values: torch.Tensor = self.network(input_values)
        return output_values


class TransformerBlock(torch.nn.Module):
    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.attention_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension=embedding_dimension,
            number_of_attention_heads=number_of_attention_heads,
            context_length=context_length,
            dropout_rate=dropout_rate,
        )
        self.feedforward_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = FeedForwardNetwork(
            embedding_dimension=embedding_dimension,
            dropout_rate=dropout_rate,
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        attention_branch = self.attention(self.attention_norm(input_values))
        values_after_attention = input_values + attention_branch
        feedforward_branch = self.feedforward(
            self.feedforward_norm(values_after_attention)
        )
        output_values: torch.Tensor = values_after_attention + feedforward_branch
        return output_values


class TinyGPT(torch.nn.Module):
    vocabulary_size: int
    context_length: int

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        number_of_attention_heads: int,
        number_of_transformer_blocks: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_length = context_length
        self.token_embedding = torch.nn.Embedding(vocabulary_size, embedding_dimension)
        self.position_embedding = torch.nn.Embedding(
            context_length, embedding_dimension
        )
        self.embedding_dropout = torch.nn.Dropout(dropout_rate)
        self.transformer_blocks = torch.nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dimension=embedding_dimension,
                    number_of_attention_heads=number_of_attention_heads,
                    context_length=context_length,
                    dropout_rate=dropout_rate,
                )
                for _ in range(number_of_transformer_blocks)
            ]
        )
        self.final_norm = torch.nn.LayerNorm(embedding_dimension)
        self.output_layer = torch.nn.Linear(embedding_dimension, vocabulary_size)

    def forward(
        self,
        input_token_ids: torch.Tensor,
        target_token_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        if input_token_ids.ndim != 2:
            raise ValueError("Inputs must have shape [batch, time].")
        batch_size, sequence_length = input_token_ids.shape
        if sequence_length > self.context_length:
            raise ValueError("Input exceeds model context length.")
        position_ids = torch.arange(sequence_length, device=input_token_ids.device)
        hidden_values = self.token_embedding(input_token_ids)
        hidden_values = hidden_values + self.position_embedding(position_ids)
        hidden_values = self.embedding_dropout(hidden_values)
        hidden_values = self.transformer_blocks(hidden_values)
        logits = self.output_layer(self.final_norm(hidden_values))
        loss = None
        if target_token_ids is not None:
            if target_token_ids.shape != input_token_ids.shape:
                raise ValueError("Targets must have the same shape as inputs.")
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(batch_size * sequence_length, self.vocabulary_size),
                target_token_ids.reshape(batch_size * sequence_length),
            )
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        input_token_ids: torch.Tensor,
        number_of_new_tokens: int,
        generator: torch.Generator,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        generated_ids = input_token_ids
        for _ in range(number_of_new_tokens):
            model_input = generated_ids[:, -self.context_length :]
            logits, _ = self(model_input)
            next_logits = logits[:, -1] / temperature
            if top_k is not None:
                effective_top_k = min(top_k, self.vocabulary_size)
                top_values, _ = torch.topk(next_logits, effective_top_k)
                next_logits = next_logits.masked_fill(
                    next_logits < top_values[:, -1, None], float("-inf")
                )
            probabilities = torch.softmax(next_logits, dim=-1)
            next_ids = torch.multinomial(
                probabilities, num_samples=1, generator=generator
            )
            generated_ids = torch.cat([generated_ids, next_ids], dim=1)
        return generated_ids

## Reuse validated config schemas

Typed configuration objects replace unchecked `int(...)` and `float(...)` coercions from arbitrary JSON values.

Architecture and format versions make compatibility explicit.

In [4]:
from dataclasses import dataclass


def require_int(
    data: dict[str, object],
    key: str,
    minimum: int,
) -> int:
    value = data.get(key)
    if type(value) is not int or value < minimum:
        raise ValueError(f"{key} must be an integer of at least {minimum}.")
    return value


def require_float(
    data: dict[str, object],
    key: str,
    minimum: float,
    maximum: float | None = None,
) -> float:
    value = data.get(key)
    if type(value) is not float or value < minimum:
        raise ValueError(f"{key} must be a floating-point value of at least {minimum}.")
    if maximum is not None and value >= maximum:
        raise ValueError(f"{key} must be less than {maximum}.")
    return value


@dataclass(frozen=True)
class ModelConfig:
    format_version: int
    architecture: str
    vocabulary_size: int
    context_length: int
    embedding_dimension: int
    number_of_attention_heads: int
    number_of_transformer_blocks: int
    dropout_rate: float

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "architecture": self.architecture,
            "vocabulary_size": self.vocabulary_size,
            "context_length": self.context_length,
            "embedding_dimension": self.embedding_dimension,
            "number_of_attention_heads": self.number_of_attention_heads,
            "number_of_transformer_blocks": self.number_of_transformer_blocks,
            "dropout_rate": self.dropout_rate,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "ModelConfig":
        expected_keys = {
            "format_version",
            "architecture",
            "vocabulary_size",
            "context_length",
            "embedding_dimension",
            "number_of_attention_heads",
            "number_of_transformer_blocks",
            "dropout_rate",
        }
        if set(data) != expected_keys:
            raise ValueError("Model config has missing or unexpected fields.")
        if data["format_version"] != 1 or data["architecture"] != "TinyGPT-v1":
            raise ValueError("Unsupported model config format or architecture.")
        config = cls(
            format_version=1,
            architecture="TinyGPT-v1",
            vocabulary_size=require_int(data, "vocabulary_size", 1),
            context_length=require_int(data, "context_length", 1),
            embedding_dimension=require_int(data, "embedding_dimension", 1),
            number_of_attention_heads=require_int(data, "number_of_attention_heads", 1),
            number_of_transformer_blocks=require_int(
                data, "number_of_transformer_blocks", 1
            ),
            dropout_rate=require_float(data, "dropout_rate", 0.0, 1.0),
        )
        if config.embedding_dimension % config.number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        return config


@dataclass(frozen=True)
class TrainingConfig:
    format_version: int
    batch_size: int
    learning_rate: float
    training_steps: int
    weight_decay: float
    max_gradient_norm: float
    model_seed: int
    training_seed: int
    dataset_description: str

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "batch_size": self.batch_size,
            "learning_rate": self.learning_rate,
            "training_steps": self.training_steps,
            "weight_decay": self.weight_decay,
            "max_gradient_norm": self.max_gradient_norm,
            "model_seed": self.model_seed,
            "training_seed": self.training_seed,
            "dataset_description": self.dataset_description,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "TrainingConfig":
        expected_keys = {
            "format_version",
            "batch_size",
            "learning_rate",
            "training_steps",
            "weight_decay",
            "max_gradient_norm",
            "model_seed",
            "training_seed",
            "dataset_description",
        }
        if set(data) != expected_keys or data["format_version"] != 1:
            raise ValueError("Unsupported training config schema.")
        description = data["dataset_description"]
        if not isinstance(description, str) or not description:
            raise ValueError("dataset_description must be a non-empty string.")
        learning_rate = require_float(data, "learning_rate", 0.0)
        max_gradient_norm = require_float(data, "max_gradient_norm", 0.0)
        if learning_rate == 0.0 or max_gradient_norm == 0.0:
            raise ValueError("learning_rate and max_gradient_norm must be positive.")
        return cls(
            format_version=1,
            batch_size=require_int(data, "batch_size", 1),
            learning_rate=learning_rate,
            training_steps=require_int(data, "training_steps", 1),
            weight_decay=require_float(data, "weight_decay", 0.0),
            max_gradient_norm=max_gradient_norm,
            model_seed=require_int(data, "model_seed", 0),
            training_seed=require_int(data, "training_seed", 0),
            dataset_description=description,
        )

## Define shared construction and file helpers

The same model factory serves the package producer and later loader.

JSON output is sorted for readability, while JSON input is validated as an object before schema parsing.

In [5]:
import json  # noqa: I001
from collections.abc import Mapping
from pathlib import Path
from typing import cast


def create_model(config: ModelConfig) -> TinyGPT:
    return TinyGPT(
        vocabulary_size=config.vocabulary_size,
        context_length=config.context_length,
        embedding_dimension=config.embedding_dimension,
        number_of_attention_heads=config.number_of_attention_heads,
        number_of_transformer_blocks=config.number_of_transformer_blocks,
        dropout_rate=config.dropout_rate,
    )


def save_json(data: Mapping[str, object], path: Path) -> None:
    serialized = json.dumps(dict(data), indent=2, ensure_ascii=False, sort_keys=True)
    path.write_text(serialized + "\n", encoding="utf-8")


def load_json_object(path: Path) -> dict[str, object]:
    loaded_value = cast(object, json.loads(path.read_text(encoding="utf-8")))
    if not isinstance(loaded_value, dict):
        raise TypeError(f"{path.name} must contain a JSON object.")
    result: dict[str, object] = {}
    for key, value in loaded_value.items():
        if not isinstance(key, str):
            raise TypeError("JSON object keys must be strings.")
        result[key] = value
    return result


def load_tensor_state_dict(path: Path) -> dict[str, torch.Tensor]:
    loaded_value: object = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(loaded_value, dict):
        raise TypeError("Model state must be a dictionary.")
    tensor_state: dict[str, torch.Tensor] = {}
    for key, value in loaded_value.items():
        if not isinstance(key, str) or not isinstance(value, torch.Tensor):
            raise TypeError("Model state must map string names to tensors.")
        tensor_state[key] = value
    return tensor_state

## Prepare a standalone saved package

This cell is test-fixture setup, not part of the later loading pipeline.

It trains for 300 deterministic updates, writes the Chapter 88 filenames, and returns no live training objects.

In [6]:
import tempfile  # noqa: I001


def create_saved_package_fixture(artifact_directory: Path) -> list[float]:
    training_text = (
        "Alice saw the rabbit. "
        "The rabbit saw Alice. "
        "Alice followed the rabbit. "
        "The rabbit was late. "
    ) * 40
    tokenizer = SaveableCharacterTokenizer()
    tokenizer.train(training_text)
    model_config = ModelConfig(
        format_version=1,
        architecture="TinyGPT-v1",
        vocabulary_size=tokenizer.vocabulary_size,
        context_length=64,
        embedding_dimension=64,
        number_of_attention_heads=4,
        number_of_transformer_blocks=2,
        dropout_rate=0.1,
    )
    training_config = TrainingConfig(
        format_version=1,
        batch_size=8,
        learning_rate=0.0003,
        training_steps=300,
        weight_decay=0.01,
        max_gradient_norm=1.0,
        model_seed=8901,
        training_seed=8902,
        dataset_description="Four fixed Alice sentences repeated 40 times.",
    )
    torch.manual_seed(training_config.model_seed)
    model = create_model(model_config)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=training_config.learning_rate,
        weight_decay=training_config.weight_decay,
    )
    token_ids = torch.tensor(tokenizer.encode(training_text), dtype=torch.long)
    batch_generator = torch.Generator().manual_seed(training_config.training_seed)
    torch.manual_seed(training_config.training_seed)
    checkpoint_losses: list[float] = []
    model.train()

    for step in range(1, training_config.training_steps + 1):
        valid_starts = token_ids.numel() - model_config.context_length
        starts = torch.randint(
            0,
            valid_starts,
            (training_config.batch_size,),
            generator=batch_generator,
        )
        offsets = torch.arange(model_config.context_length)
        indexes = starts[:, None] + offsets[None, :]
        inputs = token_ids[indexes]
        targets = token_ids[indexes + 1]
        _, loss = model(inputs, targets)
        if loss is None or not torch.isfinite(loss):
            raise RuntimeError("Fixture training loss is non-finite.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=training_config.max_gradient_norm,
            error_if_nonfinite=True,
        )
        optimizer.step()
        if step == 1 or step % 100 == 0:
            checkpoint_losses.append(float(loss.item()))

    torch.save(model.state_dict(), artifact_directory / "model_state.pt")
    save_json(tokenizer.to_dict(), artifact_directory / "tokenizer.json")
    save_json(model_config.to_dict(), artifact_directory / "model_config.json")
    save_json(training_config.to_dict(), artifact_directory / "training_config.json")
    return checkpoint_losses


temporary_directory = tempfile.TemporaryDirectory()
saved_package_directory = Path(temporary_directory.name) / "saved_tiny_gpt"
saved_package_directory.mkdir()
fixture_losses = create_saved_package_fixture(saved_package_directory)

print("Fixture batch losses:", [round(loss, 4) for loss in fixture_losses])
print(
    "Saved filenames:",
    sorted(path.name for path in saved_package_directory.iterdir()),
)
assert fixture_losses[-1] < fixture_losses[0]

Fixture batch losses: [3.2637, 1.3941, 1.0117, 0.9052]
Saved filenames: ['model_config.json', 'model_state.pt', 'tokenizer.json', 'training_config.json']


The falling fixture loss confirms that the package contains trained tensor updates rather than only random initialization.

No producer model, tokenizer, config, or optimizer is available outside the function.

## Later session begins

From this point onward, `saved_package_directory` is the only producer output used by the inference workflow.

In an application, this path would point to persistent storage and the class code above would usually be imported from modules.

## Build one reusable system loader

The loader performs operations in dependency order:

1. verify required filenames;

2. validate tokenizer and config JSON;

3. check vocabulary-size compatibility;

4. construct the architecture;

5. load tensor state strictly;

6. switch to evaluation mode; and

7. run a one-token finite-output smoke test.

The optional training config is returned for inspection but does not affect inference.

In [7]:
from dataclasses import dataclass


@dataclass(frozen=True)
class LoadedTinyGPTSystem:
    model: TinyGPT
    tokenizer: SaveableCharacterTokenizer
    model_config: ModelConfig
    training_config: TrainingConfig | None


def load_saved_tiny_gpt_system(
    artifact_directory: Path,
) -> LoadedTinyGPTSystem:
    required_filenames = ["model_state.pt", "tokenizer.json", "model_config.json"]
    missing_filenames = [
        filename
        for filename in required_filenames
        if not (artifact_directory / filename).is_file()
    ]
    if missing_filenames:
        raise FileNotFoundError(f"Missing required files: {missing_filenames!r}.")

    tokenizer = SaveableCharacterTokenizer.from_dict(
        load_json_object(artifact_directory / "tokenizer.json")
    )
    model_config = ModelConfig.from_dict(
        load_json_object(artifact_directory / "model_config.json")
    )
    if tokenizer.vocabulary_size != model_config.vocabulary_size:
        raise ValueError("Tokenizer and model-config vocabulary sizes differ.")

    training_config_path = artifact_directory / "training_config.json"
    training_config = (
        TrainingConfig.from_dict(load_json_object(training_config_path))
        if training_config_path.is_file()
        else None
    )
    model = create_model(model_config)
    state_dictionary = load_tensor_state_dict(artifact_directory / "model_state.pt")
    try:
        incompatible_keys = model.load_state_dict(state_dictionary, strict=True)
    except RuntimeError as error:
        raise RuntimeError(
            "Model state is incompatible with the saved model config and code."
        ) from error
    if incompatible_keys.missing_keys or incompatible_keys.unexpected_keys:
        raise RuntimeError("Model state did not load strictly.")
    model.eval()

    with torch.inference_mode():
        probe_logits, _ = model(torch.tensor([[0]], dtype=torch.long))
    expected_probe_shape = (1, 1, model_config.vocabulary_size)
    if tuple(probe_logits.shape) != expected_probe_shape:
        raise RuntimeError("Loaded model produced an unexpected output shape.")
    if not torch.isfinite(probe_logits).all():
        raise RuntimeError("Loaded model produced non-finite logits.")

    return LoadedTinyGPTSystem(
        model=model,
        tokenizer=tokenizer,
        model_config=model_config,
        training_config=training_config,
    )

## Load fresh objects from files

The output reports only stable package facts and does not expose the random temporary path.

In [8]:
loaded_system = load_saved_tiny_gpt_system(saved_package_directory)

print("Architecture:", loaded_system.model_config.architecture)
print("Vocabulary size:", loaded_system.tokenizer.vocabulary_size)
print("Context length:", loaded_system.model.context_length)
print("Model is in evaluation mode:", not loaded_system.model.training)
print("Training config available:", loaded_system.training_config is not None)

assert loaded_system.model.vocabulary_size == loaded_system.tokenizer.vocabulary_size
assert not loaded_system.model.training

Architecture: TinyGPT-v1
Vocabulary size: 18
Context length: 64
Model is in evaluation mode: True
Training config available: True


## Encode and inspect one prompt

Prompt IDs come only from the loaded tokenizer.

The forward shape checks that the loaded model accepts those IDs and produces one score per saved vocabulary token.

In [9]:
prompt = "Alice"
prompt_token_ids = loaded_system.tokenizer.encode(prompt)
prompt_tensor = torch.tensor([prompt_token_ids], dtype=torch.long)

with torch.inference_mode():
    prompt_logits, prompt_loss = loaded_system.model(prompt_tensor)

print("Prompt:", repr(prompt))
print("Prompt IDs:", prompt_token_ids)
print("Logit shape:", tuple(prompt_logits.shape))
print("Loss without targets:", prompt_loss)

assert tuple(prompt_logits.shape) == (
    1,
    len(prompt_token_ids),
    loaded_system.tokenizer.vocabulary_size,
)
assert prompt_loss is None

Prompt: 'Alice'
Prompt IDs: [2, 12, 11, 6, 8]
Logit shape: (1, 5, 18)
Loss without targets: None


## Generate without losing a long prompt

Only the latest `context_length` IDs condition the first generated token when a prompt is too long.

The returned text still preserves the complete user prompt by decoding only the newly sampled continuation and appending it to the original string.

An explicit `torch.Generator` keeps sampling reproducible without resetting global PyTorch randomness.

In [10]:
@dataclass(frozen=True)
class GenerationResult:
    text: str
    continuation_token_ids: list[int]
    prompt_token_count: int
    conditioning_token_count: int


@torch.inference_mode()
def generate_text_from_loaded_system(
    system: LoadedTinyGPTSystem,
    prompt: str,
    number_of_new_tokens: int,
    seed: int,
    temperature: float = 1.0,
    top_k: int | None = None,
) -> GenerationResult:
    if not prompt:
        raise ValueError("Prompt must not be empty.")
    if number_of_new_tokens < 0:
        raise ValueError("number_of_new_tokens must be nonnegative.")
    if temperature <= 0:
        raise ValueError("temperature must be positive.")
    if top_k is not None and top_k <= 0:
        raise ValueError("top_k must be positive when provided.")

    system.model.eval()
    prompt_ids = system.tokenizer.encode(prompt)
    conditioning_ids = prompt_ids[-system.model.context_length :]
    input_ids = torch.tensor([conditioning_ids], dtype=torch.long)
    generator = torch.Generator().manual_seed(seed)
    generated_ids = system.model.generate(
        input_ids,
        number_of_new_tokens=number_of_new_tokens,
        generator=generator,
        temperature=temperature,
        top_k=top_k,
    )
    continuation_ids = generated_ids[0, len(conditioning_ids) :].tolist()
    continuation = system.tokenizer.decode(continuation_ids)
    return GenerationResult(
        text=prompt + continuation,
        continuation_token_ids=continuation_ids,
        prompt_token_count=len(prompt_ids),
        conditioning_token_count=len(conditioning_ids),
    )

## Generate from the loaded model

The output includes the original prompt plus 100 sampled character tokens.

In [11]:
generated_result = generate_text_from_loaded_system(
    loaded_system,
    prompt="Alice",
    number_of_new_tokens=100,
    seed=8903,
    temperature=1.0,
    top_k=10,
)

print("Generated text:")
print(generated_result.text)
print("Continuation token count:", len(generated_result.continuation_token_ids))
assert generated_result.text.startswith("Alice")
assert len(generated_result.continuation_token_ids) == 100

Generated text:
Alice rabitce foce. Alit. lice  Thed te saw we sabit. rabbit. sat. rabithe Alow Alabithe. t. saw ras. Awd
Continuation token count: 100


The sample is intentionally small-model output; successful generation demonstrates package usability rather than language quality.

## Verify sampling reproducibility

The same explicit seed reproduces the continuation, while a different seed normally changes it.

In [12]:
repeated_result = generate_text_from_loaded_system(
    loaded_system, "Alice", 100, seed=8903, top_k=10
)
different_seed_result = generate_text_from_loaded_system(
    loaded_system, "Alice", 100, seed=8904, top_k=10
)

print(
    "Same seed gives same continuation:",
    repeated_result.continuation_token_ids == generated_result.continuation_token_ids,
)
print(
    "Different seed changes continuation:",
    different_seed_result.continuation_token_ids
    != generated_result.continuation_token_ids,
)

assert repeated_result.continuation_token_ids == generated_result.continuation_token_ids
assert (
    different_seed_result.continuation_token_ids
    != generated_result.continuation_token_ids
)

Same seed gives same continuation: True
Different seed changes continuation: True


## Verify long-prompt cropping behavior

The model conditions on only 64 recent characters, but the result preserves the complete 120-character prompt before its continuation.

In [13]:
long_prompt = "Alice " * 20
long_prompt_result = generate_text_from_loaded_system(
    loaded_system,
    prompt=long_prompt,
    number_of_new_tokens=20,
    seed=8905,
    top_k=10,
)

print("Original prompt tokens:", long_prompt_result.prompt_token_count)
print("Conditioning tokens used:", long_prompt_result.conditioning_token_count)
print(
    "Full prompt preserved in result:", long_prompt_result.text.startswith(long_prompt)
)

assert long_prompt_result.prompt_token_count == 120
assert long_prompt_result.conditioning_token_count == 64
assert long_prompt_result.text.startswith(long_prompt)

Original prompt tokens: 120
Conditioning tokens used: 64
Full prompt preserved in result: True


Cropping is an inference policy, not a way to extend the model's trained context length.

The earliest prompt characters remain visible in returned text but do not influence the generated continuation.

## Handle tokenizer limits clearly

This character tokenizer has no unknown-token fallback, so an unseen character must fail before model execution.

In [14]:
try:
    generate_text_from_loaded_system(
        loaded_system,
        prompt="Alice!",
        number_of_new_tokens=5,
        seed=8906,
    )
except ValueError as error:
    print("Expected encoding error:", error)
else:
    raise RuntimeError("Unknown-character prompt unexpectedly encoded.")

Expected encoding error: Unknown tokens: ['!'].


The error is a tokenizer-vocabulary limitation rather than a loading failure.

## Diagnose architecture mismatch

Changing the saved embedding dimension still produces a valid-looking config, but its newly constructed tensor shapes cannot accept the saved state.

The loader converts PyTorch's long size-mismatch report into one stable package-level error while preserving the original exception as its cause.

In [15]:
import shutil  # noqa: I001


bad_architecture_directory = Path(temporary_directory.name) / "bad_architecture"
shutil.copytree(saved_package_directory, bad_architecture_directory)
bad_config_data = load_json_object(bad_architecture_directory / "model_config.json")
bad_config_data["embedding_dimension"] = 128
save_json(bad_config_data, bad_architecture_directory / "model_config.json")

try:
    load_saved_tiny_gpt_system(bad_architecture_directory)
except RuntimeError as error:
    print("Expected architecture error:", error)
else:
    raise RuntimeError("Mismatched architecture unexpectedly loaded.")

Expected architecture error: Model state is incompatible with the saved model config and code.


Architecture mismatch usually fails loudly because state names or tensor shapes differ.

## Show the limit of a vocabulary-size check

A permuted tokenizer can retain the same vocabulary size and valid contiguous IDs.

Without a manifest, checksum, or saved sanity mapping that binds files together, the loader cannot know that this different tokenizer came from another package.

In [16]:
wrong_tokenizer_directory = Path(temporary_directory.name) / "wrong_tokenizer"
shutil.copytree(saved_package_directory, wrong_tokenizer_directory)
correct_tokenizer_data = load_json_object(wrong_tokenizer_directory / "tokenizer.json")
correct_mapping = correct_tokenizer_data["token_to_id"]
if not isinstance(correct_mapping, dict):
    raise RuntimeError("Fixture tokenizer mapping is malformed.")
reversed_mapping = {
    token: len(correct_mapping) - 1 - token_id
    for token, token_id in correct_mapping.items()
    if isinstance(token, str) and type(token_id) is int
}
wrong_tokenizer_data = {
    "format_version": 1,
    "tokenizer_type": "character",
    "token_to_id": reversed_mapping,
}
save_json(wrong_tokenizer_data, wrong_tokenizer_directory / "tokenizer.json")
wrong_system = load_saved_tiny_gpt_system(wrong_tokenizer_directory)

correct_ids = loaded_system.tokenizer.encode("Alice")
wrong_ids = wrong_system.tokenizer.encode("Alice")
print(
    "Vocabulary sizes match:",
    wrong_system.model.vocabulary_size == loaded_system.model.vocabulary_size,
)
print("Correct IDs:", correct_ids)
print("Wrong IDs:  ", wrong_ids)
print("Prompt IDs match:", correct_ids == wrong_ids)

assert wrong_system.model.vocabulary_size == loaded_system.model.vocabulary_size
assert correct_ids != wrong_ids

Vocabulary sizes match: True
Correct IDs: [2, 12, 11, 6, 8]
Wrong IDs:   [15, 5, 6, 11, 9]
Prompt IDs match: False


Successful loading is therefore necessary but not sufficient evidence that independently supplied files belong together.

Artifact manifests with hashes can strengthen package integrity, while authenticated signatures are needed when provenance must be trusted.

## What generation proves

This execution demonstrates that:

- required files were readable;

- JSON schemas and vocabulary size were valid;

- architecture construction and strict tensor loading succeeded;

- evaluation produced finite scores of the expected shape; and

- the tokenizer encoded and decoded generated IDs.

It does not establish model quality, generalization, safe behavior, or artifact provenance.

## Clean up all generated fixtures

Cleanup removes the valid package and both mismatch copies so notebook execution leaves no artifacts in the repository or temporary storage.

In [17]:
temporary_root = Path(temporary_directory.name)
temporary_directory.cleanup()

print("Temporary package tree removed:", not temporary_root.exists())
assert not temporary_root.exists()

Temporary package tree removed: True


## Verify chapter contracts

The final checks cover loading mode, package metadata, prompt encoding, generation length, seed control, context cropping, mismatch failures, and cleanup.

In [18]:
assert not loaded_system.model.training
assert loaded_system.model_config.architecture == "TinyGPT-v1"
assert loaded_system.training_config is not None
assert loaded_system.tokenizer.encode("Alice") == prompt_token_ids
assert len(generated_result.continuation_token_ids) == 100
assert repeated_result.continuation_token_ids == generated_result.continuation_token_ids
assert long_prompt_result.conditioning_token_count == loaded_system.model.context_length
assert correct_ids != wrong_ids
assert not temporary_root.exists()
print("All later-loading and generation contracts passed.")

All later-loading and generation contracts passed.


## Common mistakes

- Assuming a previous notebook left files behind makes the new notebook depend on hidden execution history.

- Using stale filenames or model attribute names breaks continuity with the package producer.

- Loading state without validated config values turns corrupt metadata into confusing constructor errors.

- Omitting `weights_only=True` uses a less restricted pickle-loading path than tensor state requires.

- Forgetting `model.eval()` leaves dropout active during inference.

- Calling `torch.manual_seed` inside a generation helper mutates global random state.

- Decoding only the cropped conditioning prefix silently removes the beginning of a long prompt from returned text.

- Checking vocabulary size alone cannot detect a same-size tokenizer permutation.

- Treating successful generation as a quality or security evaluation overclaims the smoke test.

- Assuming training config enables exact resume ignores optimizer, scheduler, RNG, and sampler state.

## Takeaways

- Later inference reconstructs a system from compatible code, tokenizer data, model config, and tensor state.

- A reusable loader should validate files and schemas before model construction, then load state strictly and run a smoke test.

- `map_location='cpu'`, `weights_only=True`, trusted sources, and evaluation mode are appropriate CPU inference defaults.

- Explicit generators make sampling repeatable without disturbing global randomness.

- Long prompts can be cropped for conditioning while the returned string preserves the complete original prompt.

- Architecture mismatch usually fails loudly, but a same-size tokenizer mismatch can run silently.

- Generation proves mechanical package usability, not language quality or artifact provenance.

## What comes next

Loading restores fixed inference state, while resumable checkpointing must also restore optimizer progress and select which training snapshot to keep.

A best-validation-checkpoint workflow can build directly on this chapter's strict loader and smoke tests.